# Inter-code residuals vs planet parameters

For every planet the database has a noiseless spectrum from each radiative-transfer code (POSEIDON, petitRADTRANS, TauREx) with **identical parameters**. For a pair of codes $(a, b)$ the difference spectrum $\mathbf{y}^{(a)} - \mathbf{y}^{(b)}$ of the same planet isolates the code systematics. This notebook summarises that difference with one number per planet and plots it against the planet parameters.

| metric | definition |
|---|---|
| `mse` | raw MSE, $\frac{1}{D}\sum_j (y^{(a)}_j - y^{(b)}_j)^2$ in ppm², no normalisation |
| `zmse` | MSE between the two spectra after each is z-normalised by its own mean and std (planet-normalised) |
| `rms` | RMS of the difference after removing its wavelength mean (shape residual, ppm) |
| `offset` | wavelength mean of the difference (ppm) |
| `relative` | `rms` divided by the pair's mean feature height (%) |

Derived quantities: surface gravity $g = G M_p / R_p^2$ (m/s²), $\log_{10} g$ in Jupiter units, scale height $H = k_B T / (\mu\, m_H\, g)$ (km), feature height $\max - \min$ of the spectrum (ppm).

**Standalone:** needs only `numpy` and `matplotlib`, and reads the `.npy` files in the data directory. Change the settings in the **Configuration** cell, then *Run All*. Every plotting function also takes keyword arguments to override settings for a single plot.

In [ ]:
from pathlib import Path
import json

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker

%matplotlib inline

## Configuration
Everything you are likely to change is here.

In [ ]:
# --- data -----------------------------------------------------------------
DATA_DIR = Path("data")            # folder with <code>_spec_binned.npy, <code>_params_clean.npy, wavelength_binned.npy
CODES = {"pos": "POSEIDON", "prt": "petitRADTRANS", "taurex": "TauREx"}   # file prefix -> label
PAIRS = None                       # None = all pairs, or e.g. [("pos", "taurex")]

# --- planet selection ---------------------------------------------------
APPLY_FH_CUT = False               # True: keep planets whose feature height >= MIN_FH_PPM in every code
MIN_FH_PPM = 210.0
T_MIN, T_MAX = None, 1500.0        # temperature range in K (None = no limit); T_MIN <= T < T_MAX
EXTRA_SELECTION = None             # optional function q -> boolean mask, e.g. lambda q: q["Rp"] > 0.5

# --- physics --------------------------------------------------------------
MU = 2.3                           # mean molecular weight used for the scale height H

# --- what to plot ---------------------------------------------------------
METRICS_TO_PLOT = ["mse", "zmse"]  # any of: mse, zmse, rms, offset, relative
COLUMNS = ["Rs", "Rp", "Mp", "T", "H2O", "CH4", "CO", "CO2", "NH3", "g", "H"]
# available columns: Rs Rp Mp T H2O CH4 CO CO2 NH3 g log_g H fh

LOG_X_COLUMNS = []                 # columns drawn on a log-scaled x axis, e.g. ["g", "H"]
CLIP_X = {"g": 99, "H": 99}        # column -> upper x-limit percentile (ignored for LOG_X_COLUMNS); {} = no clipping
LOG_Y = {"mse": True, "zmse": True, "rms": True, "relative": True, "offset": "symlog"}

# --- plot style -----------------------------------------------------------
N_BINS = 20                        # quantile bins for the percentile curves
BAND = (16, 84)                    # shaded percentile band
UPPER_LINE = 90                    # dashed percentile line (None to hide)
SHOW_BINNED = True                 # draw median / band / percentile line
POINT_SIZE, POINT_ALPHA, POINT_COLOR = 1.0, 0.15, "grey"
MAX_POINTS = None                  # randomly subsample the scatter for speed (None = all points)
SEED = 0
PANEL_W, PANEL_H = 2.3, 2.5        # inches per panel
SHARE_Y = "row"                    # "row", "all" or False
DPI = 110

# --- saving ---------------------------------------------------------------
SAVE_DIR = None                    # e.g. Path("runs/residuals/notebook") to save figures as PNG
SAVE_DPI = 150

## Constants and labels

In [ ]:
PARAMS = ["Rs", "Rp", "Mp", "T", "H2O", "CH4", "CO", "CO2", "NH3"]   # column order in the params files

AXIS_LABELS = {
    "Rs": "Rs (R☉)", "Rp": "Rp (R_J)", "Mp": "Mp (M_J)", "T": "T (K)",
    "H2O": "log X(H₂O)", "CH4": "log X(CH₄)", "CO": "log X(CO)", "CO2": "log X(CO₂)", "NH3": "log X(NH₃)",
    "g": "g (m/s²)", "log_g": "log g (Mp/Rp², Jupiter units)", "H": "scale height H (km)",
    "fh": "mean feature height (ppm)",
}
METRIC_LABELS = {
    "mse": "raw MSE (ppm²)", "zmse": "z-normalised MSE", "rms": "shape residual RMS (ppm)",
    "offset": "mean offset (ppm)", "relative": "RMS / feature height (%)",
}

G = 6.674e-11                      # m^3 kg^-1 s^-2
M_JUP, R_JUP = 1.898e27, 7.1492e7  # kg, m (equatorial radius)
K_B, M_H = 1.380649e-23, 1.6735575e-27

## Load data

In [ ]:
def load_code(prefix, data_dir=DATA_DIR):
    spectra = np.load(Path(data_dir) / f"{prefix}_spec_binned.npy")     # transit depth (fraction)
    params = np.load(Path(data_dir) / f"{prefix}_params_clean.npy")
    return spectra, params


spectra_all, params_all = {}, None
for code_name in CODES:
    s, p = load_code(code_name)
    if params_all is not None and not np.array_equal(p, params_all):
        raise ValueError(f"parameters of {code_name} differ from the first code: rows are not the same planets")
    spectra_all[code_name], params_all = s * 1e6, p                    # spectra in ppm
wavelength = np.load(DATA_DIR / "wavelength_binned.npy")              # micron
print(f"{len(params_all)} planets, {spectra_all[next(iter(CODES))].shape[1]} wavelength bins "
      f"({wavelength.min():.2f}-{wavelength.max():.2f} µm), codes: {', '.join(CODES.values())}")

## Derived quantities

In [ ]:
def feature_height(spectra):
    return spectra.max(axis=1) - spectra.min(axis=1)


def znormalise(spectra):
    mean = spectra.mean(axis=1, keepdims=True)
    std = spectra.std(axis=1, keepdims=True)
    return (spectra - mean) / std


def planet_quantities(params, spectra, mu=MU):
    """Dict of per-planet quantities: the 9 parameters plus g, log_g, H and mean feature height."""
    q = {name: params[:, j] for j, name in enumerate(PARAMS)}
    q["g"] = G * q["Mp"] * M_JUP / (q["Rp"] * R_JUP) ** 2
    q["log_g"] = np.log10(q["Mp"] / q["Rp"] ** 2)
    q["H"] = K_B * q["T"] / (mu * M_H * q["g"]) / 1e3
    q["fh"] = np.mean([feature_height(s) for s in spectra.values()], axis=0)
    return q


q_all = planet_quantities(params_all, spectra_all)

## Select planets

In [ ]:
def select(q, spectra, apply_fh_cut=APPLY_FH_CUT, min_fh_ppm=MIN_FH_PPM, t_min=T_MIN, t_max=T_MAX,
           extra=EXTRA_SELECTION):
    keep = np.ones(len(q["T"]), bool)
    if apply_fh_cut:
        for s in spectra.values():
            keep &= feature_height(s) >= min_fh_ppm
    if t_min is not None:
        keep &= q["T"] >= t_min
    if t_max is not None:
        keep &= q["T"] < t_max
    if extra is not None:
        keep &= extra(q)
    return keep


keep = select(q_all, spectra_all)
spectra = {c: s[keep] for c, s in spectra_all.items()}
q = {k: v[keep] for k, v in q_all.items()}
SELECTION_TEXT = ", ".join(filter(None, [
    f"feature height ≥ {MIN_FH_PPM:g} ppm in all codes" if APPLY_FH_CUT else "no feature-height cut",
    f"T ≥ {T_MIN:g} K" if T_MIN is not None else "",
    f"T < {T_MAX:g} K" if T_MAX is not None else "",
    "extra selection" if EXTRA_SELECTION is not None else "",
]))
print(f"{keep.sum()} of {len(keep)} planets kept ({SELECTION_TEXT})")

## Residual metrics per code pair

In [ ]:
def pair_metrics(sa, sb):
    """Per-planet residual metrics for two arrays of spectra in ppm, shape (n, bins)."""
    diff = sa - sb
    offset = diff.mean(axis=1)
    rms = np.sqrt(((diff - offset[:, None]) ** 2).mean(axis=1))
    fh = (feature_height(sa) + feature_height(sb)) / 2
    return {
        "mse": (diff ** 2).mean(axis=1),
        "zmse": ((znormalise(sa) - znormalise(sb)) ** 2).mean(axis=1),
        "rms": rms,
        "offset": offset,
        "relative": 100 * rms / fh,
    }


codes = list(CODES)
pairs = PAIRS or [(a, b) for i, a in enumerate(codes) for b in codes[i + 1:]]
metrics = {(a, b): pair_metrics(spectra[a], spectra[b]) for a, b in pairs}
print("pairs:", ", ".join(f"{CODES[a]} − {CODES[b]}" for a, b in pairs))

## Helper functions

In [ ]:
def ranks(x):
    r = np.empty(len(x))
    r[np.argsort(x)] = np.arange(len(x))
    return r


def spearman(x, y):
    return float(np.corrcoef(ranks(x), ranks(y))[0, 1])


def binned_percentiles(x, y, n_bins=N_BINS, qs=(BAND[0], 50, BAND[1], UPPER_LINE)):
    """Percentiles of y in quantile bins of x. Returns bin centres (median x) and {q: values}."""
    qs = [v for v in qs if v is not None]
    edges = np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)))
    which = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, len(edges) - 2)
    centres = np.array([np.median(x[which == i]) for i in range(len(edges) - 1)])
    pct = np.array([np.percentile(y[which == i], qs) for i in range(len(edges) - 1)])
    return centres, {v: pct[:, k] for k, v in enumerate(qs)}


def set_y_scale(ax, metric, log_y=LOG_Y):
    mode = log_y.get(metric, False)
    if mode == "symlog":
        ax.set_yscale("symlog", linthresh=10)
        ax.axhline(0, color="k", lw=0.6)
    elif mode:
        ax.set_yscale("log")


def set_x_scale(ax, column, x, log_x_columns=LOG_X_COLUMNS, clip_x=CLIP_X):
    if column in log_x_columns:
        ax.set_xscale("log")
    elif column in clip_x and clip_x[column] is not None:
        ax.set_xlim(np.min(x), np.percentile(x, clip_x[column]))


def subsample(n, max_points=MAX_POINTS, seed=SEED):
    if max_points is None or n <= max_points:
        return np.arange(n)
    return np.random.default_rng(seed).choice(n, max_points, replace=False)


def draw_panel(ax, x, y, column, metric, **style):
    s = {"n_bins": N_BINS, "show_binned": SHOW_BINNED, "point_size": POINT_SIZE, "point_alpha": POINT_ALPHA,
         "point_color": POINT_COLOR, "max_points": MAX_POINTS, "log_x_columns": LOG_X_COLUMNS, "clip_x": CLIP_X,
         "log_y": LOG_Y, **style}
    idx = subsample(len(x), s["max_points"])
    ax.scatter(x[idx], y[idx], s=s["point_size"], color=s["point_color"], alpha=s["point_alpha"], lw=0,
               rasterized=True)
    if s["show_binned"]:
        centres, pct = binned_percentiles(x, y, s["n_bins"])
        ax.fill_between(centres, pct[BAND[0]], pct[BAND[1]], color="C0", alpha=0.3, lw=0)
        ax.plot(centres, pct[50], color="k", lw=1.3)
        if UPPER_LINE is not None:
            ax.plot(centres, pct[UPPER_LINE], color="C3", lw=1, ls="--")
    set_y_scale(ax, metric, s["log_y"])
    set_x_scale(ax, column, x, s["log_x_columns"], s["clip_x"])
    ax.text(0.03, 0.95, f"ρ={spearman(x, y):+.2f}", transform=ax.transAxes, fontsize=7, va="top",
            bbox=dict(facecolor="w", alpha=0.7, lw=0))


def legend_handles():
    h = [plt.Line2D([], [], color="k", lw=1.3, label="median"),
         plt.Rectangle((0, 0), 1, 1, color="C0", alpha=0.3, label=f"{BAND[0]}–{BAND[1]}%")]
    if UPPER_LINE is not None:
        h.append(plt.Line2D([], [], color="C3", lw=1, ls="--", label=f"{UPPER_LINE}th percentile"))
    return h


def save(fig, name):
    if SAVE_DIR is not None:
        Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(SAVE_DIR) / name, dpi=SAVE_DPI, bbox_inches="tight")

In [ ]:
def plot_scatter_grid(metric, columns=None, pairs_=None, panel_w=PANEL_W, panel_h=PANEL_H, share_y=SHARE_Y,
                      dpi=DPI, title=True, **style):
    """One row per code pair, one column per quantity. Extra keyword arguments go to draw_panel."""
    columns = columns or COLUMNS
    pairs_ = pairs_ or pairs
    fig, axes = plt.subplots(len(pairs_), len(columns), figsize=(panel_w * len(columns), panel_h * len(pairs_)),
                             squeeze=False, sharey=share_y, dpi=dpi)
    for i, (a, b) in enumerate(pairs_):
        y = metrics[(a, b)][metric]
        for j, c in enumerate(columns):
            ax = axes[i, j]
            draw_panel(ax, q[c], y, c, metric, **style)
            ax.tick_params(labelsize=6)
            if i == len(pairs_) - 1:
                ax.set_xlabel(AXIS_LABELS[c], fontsize=7)
            if j == 0:
                ax.set_ylabel(f"{CODES[a]} − {CODES[b]}\n{METRIC_LABELS[metric]}", fontsize=7)
    if style.get("show_binned", SHOW_BINNED):
        fig.legend(handles=legend_handles(), loc="upper right", ncol=3, fontsize=8)
    if title:
        fig.suptitle(f"{METRIC_LABELS[metric]} between codes vs planet quantities "
                     f"({len(next(iter(q.values())))} planets; {SELECTION_TEXT}; ρ = Spearman)", fontsize=10, x=0.4)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save(fig, f"residual_scatter_{metric}.png")
    return fig


def plot_single(a, b, column, metric, figsize=(5, 4), dpi=DPI, color_by=None, cmap="viridis", **style):
    """One pair, one quantity. color_by: optional quantity name to colour the points by (no binned curves then)."""
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    x, y = q[column], metrics[(a, b)][metric]
    if color_by is None:
        draw_panel(ax, x, y, column, metric, **{"point_size": 3, "point_alpha": 0.3, **style})
        if style.get("show_binned", SHOW_BINNED):
            ax.legend(handles=legend_handles(), fontsize=7, loc="lower right")
    else:
        idx = subsample(len(x), style.get("max_points", MAX_POINTS))
        sc = ax.scatter(x[idx], y[idx], c=q[color_by][idx], s=style.get("point_size", 3), cmap=cmap, lw=0,
                        alpha=style.get("point_alpha", 0.6), rasterized=True)
        fig.colorbar(sc, ax=ax, label=AXIS_LABELS[color_by])
        set_y_scale(ax, metric, style.get("log_y", LOG_Y))
        set_x_scale(ax, column, x, style.get("log_x_columns", LOG_X_COLUMNS), style.get("clip_x", CLIP_X))
        ax.text(0.03, 0.95, f"ρ={spearman(x, y):+.2f}", transform=ax.transAxes, fontsize=8, va="top",
                bbox=dict(facecolor="w", alpha=0.7, lw=0))
    ax.set(xlabel=AXIS_LABELS[column], ylabel=METRIC_LABELS[metric], title=f"{CODES[a]} − {CODES[b]}")
    fig.tight_layout()
    save(fig, f"single_{a}-{b}_{metric}_vs_{column}" + (f"_by_{color_by}" if color_by else "") + ".png")
    return fig


def plot_map(metric, x_col="T", y_col="g", n_bins=15, min_count=20, stat=np.median, log_x=False, log_y=True,
             log_color=True, cmap="magma_r", dpi=DPI):
    """Statistic (default median) of a metric in 2D bins of two quantities, one panel per pair."""
    def edges(v, log):
        lo, hi = np.percentile(v, 0.5), np.percentile(v, 99.5)
        return np.geomspace(lo, hi, n_bins + 1) if log else np.linspace(lo, hi, n_bins + 1)
    x, y = q[x_col], q[y_col]
    xe, ye = edges(x, log_x), edges(y, log_y)
    xi = np.searchsorted(xe, x, side="right") - 1
    yi = np.searchsorted(ye, y, side="right") - 1
    inside = (xi >= 0) & (xi < n_bins) & (yi >= 0) & (yi < n_bins)
    maps = []
    for a, b in pairs:
        m = metrics[(a, b)][metric]
        grid = np.full((n_bins, n_bins), np.nan)
        for p in range(n_bins):
            for r in range(n_bins):
                sel = inside & (xi == p) & (yi == r)
                if sel.sum() >= min_count:
                    grid[p, r] = stat(m[sel])
        maps.append(grid)
    finite = np.concatenate([g[np.isfinite(g)] for g in maps])
    norm = (matplotlib.colors.LogNorm(finite[finite > 0].min(), finite.max()) if log_color
            else matplotlib.colors.Normalize(finite.min(), finite.max()))
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.8 * len(pairs), 4), squeeze=False, dpi=dpi)
    for ax, (a, b), grid in zip(axes[0], pairs, maps):
        im = ax.pcolormesh(xe, ye, grid.T, cmap=cmap, norm=norm)
        ax.set(xlabel=AXIS_LABELS[x_col], title=f"{CODES[a]} − {CODES[b]}")
        if log_x:
            ax.set_xscale("log")
        if log_y:
            ax.set_yscale("log")
    axes[0, 0].set_ylabel(AXIS_LABELS[y_col])
    fig.colorbar(im, ax=axes[0].tolist(), label=f"{getattr(stat, '__name__', 'stat')} {METRIC_LABELS[metric]}")
    save(fig, f"map_{metric}_{x_col}_{y_col}.png")
    return fig


def plot_examples(a, b, metric, n=4, which="largest", dpi=DPI):
    """Spectra of all codes and the a − b difference for the planets with the largest/smallest metric."""
    m = metrics[(a, b)][metric]
    order = np.argsort(m)
    picks = order[::-1][:n] if which == "largest" else order[:n]
    wl = np.argsort(wavelength)
    fig, axes = plt.subplots(2, n, figsize=(3.6 * n, 5.2), sharex=True, squeeze=False, dpi=dpi,
                             gridspec_kw={"height_ratios": [2, 1]})
    for k, i in enumerate(picks):
        for c_i, c in enumerate(codes):
            axes[0, k].plot(wavelength[wl], spectra[c][i, wl], color=f"C{c_i}", lw=1, label=CODES[c])
        axes[1, k].plot(wavelength[wl], (spectra[a][i] - spectra[b][i])[wl], "k.-", lw=1, ms=3)
        axes[1, k].axhline(0, color="grey", lw=0.6)
        axes[0, k].set_title(f"{metric}={m[i]:.3g}\nT={q['T'][i]:.0f} K, g={q['g'][i]:.1f} m/s², H={q['H'][i]:.0f} km",
                             fontsize=8)
        axes[1, k].set_xlabel("wavelength (µm)")
        axes[1, k].set_xscale("log")
        axes[1, k].set_xticks([0.6, 1, 2, 3, 5, 7], ["0.6", "1", "2", "3", "5", "7"])
        axes[1, k].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    axes[0, 0].set_ylabel("transit depth (ppm)")
    axes[1, 0].set_ylabel("difference (ppm)")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(f"{which} {metric}: {CODES[a]} − {CODES[b]} (bottom: {CODES[a]} − {CODES[b]})")
    fig.tight_layout()
    save(fig, f"examples_{a}-{b}_{metric}_{which}.png")
    return fig

## Summary statistics

In [ ]:
def summary(metrics_list=("mse", "zmse", "rms", "offset"), columns=None):
    columns = columns or COLUMNS
    for a, b in pairs:
        print(f"=== {CODES[a]} − {CODES[b]} ===")
        for k in metrics_list:
            v = metrics[(a, b)][k]
            print(f"  {k:>8}: mean {v.mean():.3g}  median {np.median(v):.3g}  "
                  f"p16 {np.percentile(v, 16):.3g}  p84 {np.percentile(v, 84):.3g}  p99 {np.percentile(v, 99):.3g}")
        print("  Spearman ρ with:   " + " ".join(f"{c:>7}" for c in columns))
        for k in metrics_list:
            print(f"  {k:>17}  " + " ".join(f"{spearman(q[c], metrics[(a, b)][k]):>+7.2f}" for c in columns))
        print()


summary()

## Scatter grids
One figure per metric in `METRICS_TO_PLOT`. To change a single figure without touching the configuration, pass options directly, for example:
```python
plot_scatter_grid("zmse", columns=["T", "H2O", "g", "H"], log_x_columns=["g", "H"], point_alpha=0.05)
```

In [ ]:
for metric in METRICS_TO_PLOT:
    plot_scatter_grid(metric)
    plt.show()

## Variant: log-scaled g and H axes
The same quantities g and H (not log g), drawn on log-scaled axes to show their full range.

In [ ]:
for metric in METRICS_TO_PLOT:
    plot_scatter_grid(metric, log_x_columns=["g", "H"])
    plt.show()

## Single panel
Zoom into one pair and one quantity. `color_by` colours the points by another quantity, to look for a second driver.

In [ ]:
plot_single("pos", "taurex", "H2O", "zmse")
plt.show()
plot_single("pos", "prt", "T", "zmse", color_by="CH4")
plt.show()

## 2D map
Median of a metric in bins of two quantities. Set `log_x` / `log_y` to match the quantities, and `stat=np.mean` (or any function) to change the statistic.

In [ ]:
plot_map("zmse", x_col="T", y_col="g", log_y=True)
plt.show()
plot_map("mse", x_col="T", y_col="H", log_y=True)
plt.show()

## Example spectra
The planets where a pair differs most (or least) under a given metric: the three codes' spectra (top) and the difference (bottom).

In [ ]:
plot_examples("pos", "prt", "zmse", n=4, which="largest")
plt.show()